In [4]:
import pandas as pd
import sklearn.metrics
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [7]:
primevul_processed = pd.read_csv("../../result/primevul.csv")
magma = pd.read_csv('../../result/magma_all_mcs.csv')
magma['dataset'] = 'magma'
magma['scaled_first_token_var'] = -np.exp(np.abs(magma.first_token_var - np.percentile(magma.first_token_var, 50)) / 40)
magma["neg_exact_match"] = -magma["exact_match"]

scaler = MinMaxScaler().fit(magma[["loss", "neg_exact_match", "scaled_first_token_var", "mask_ast_complexity"]])
primevul_processed["scaled_first_token_var"] = -np.exp(np.abs(primevul_processed.first_token_var - np.percentile(magma.first_token_var, 50)) / 40)
primevul_processed["neg_exact_match"] = -primevul_processed["exact_match"]
primevul_processed["new_score"] = scaler.transform(primevul_processed[["loss", "neg_exact_match", "scaled_first_token_var", "mask_ast_complexity"]]).sum(axis=1)

MFRs = []
def calculate_ranked_metrics(df):
    # MFRs = []
    global MFRs

    df_grouped = df.groupby(['file','func_range'])
    for name, group in df_grouped:
        group = group.sample(frac=1, random_state=0).reset_index(drop=True)
        if group.vul_label.sum() == 0:
            continue
        
        group = group.sort_values(["new_score", "vul_label"], ascending=[False, True])
        group = group.reset_index(drop=True)

        for (idx, line) in group.iterrows():
            if line.vul_label == 1:
                MFRs.append(idx)
                break

    MFR = np.mean(MFRs)
    N_MFR = MFR / 141
    Top_1 = (np.array(MFRs) <= 0).sum() / len(MFRs)
    Top_3 = (np.array(MFRs) <= 2).sum() / len(MFRs)
    Top_5 = (np.array(MFRs) <= 4).sum() / len(MFRs)
    Top_10 = (np.array(MFRs) <= 9).sum() / len(MFRs)

    return f"{Top_1*100:.1f}\% & {Top_3*100:.1f}\% & {Top_5*100:.1f}\% & {MFR:.1f} & {N_MFR:.2f}"

print(f"{sklearn.metrics.roc_auc_score(primevul_processed['vul_label'], primevul_processed['new_score'])*100:.1f}\% & {calculate_ranked_metrics(primevul_processed)} ")

61.8\% & 11.7\% & 31.2\% & 41.4\% & 25.0 & 0.18 
